# 04 — Acquire and scan complete pilot captures
Use a CPU runtime. This notebook downloads all 24 manifest captures (about 1.13 GB uncompressed source data) as individual compressed archives, scans every row in chunks, and compares packet fingerprints across files. It saves progress after each file. Re-running skips verified downloads and completed scans.

Maximum download is 128 MiB per archive, at most 3 GiB for 24 files; actual compressed size should be smaller. Fingerprints also consume Drive storage. No training occurs. Keep the unresolved license recorded: this acquisition does not establish publication or redistribution rights. Raw addresses are never printed or committed to GitHub.

A match in naive timestamp ranges is a review candidate, not proof of shared sessions. Packet fingerprints are 64-bit candidate-duplicate checks; no match does not prove independent sessions or devices. This is a development data audit, not test-set performance evaluation.


Scanner v2 preserves undecodable UTF-8 bytes as visible hex escapes and counts them. No packet rows are dropped for decoding. Verified v1 scans are reused because they completed strict UTF-8 decoding. Existing downloaded archives are retained.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, datetime, hashlib
PROJECT=Path('/content/drive/MyDrive/5G_QoS_Research')
plans=sorted((PROJECT/'data/manifests').glob('capture_plan_*/capture_split_manifest.json'))
if not plans:raise RuntimeError('Run notebook 03 first.')
PLAN=plans[-1]
rows=json.loads(PLAN.read_text())
if len(rows)!=24 or len({r['source_file'] for r in rows})!=24:
    raise RuntimeError('Expected the verified 24-file development manifest.')
expected='0c69e31197b785aa76b491c3384d093fd74c465e83ebb03424388d9975781eb1'
actual=hashlib.sha256(json.dumps(rows,sort_keys=True).encode()).hexdigest()
if actual!=expected:raise RuntimeError('Manifest differs from verified pilot. Review changes first.')
ROOT=PROJECT/'data/full_capture_audit_v1'
ARCHIVES=ROOT/'archives';SCANS=ROOT/'scans'
ROOT.mkdir(parents=True,exist_ok=True)
print('Using manifest:', PLAN)
print('Captures:',len(rows),'uncompressed GB:',sum(r['source_bytes'] for r in rows)/1e9)


In [ ]:
"""Resumable file acquisition and bounded-memory capture diagnostics, version 1."""
import hashlib, json, urllib.request, urllib.parse, zipfile, os, codecs
from pathlib import Path
import numpy as np
import pandas as pd

SCAN_VERSION=2

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def acquire_capture(row, root, cap=128*1024*1024):
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    archive=root/(row['capture_id']+'.zip');receipt=archive.with_suffix('.receipt.json')
    if archive.exists() and receipt.exists():
        info=json.loads(receipt.read_text())
        if info['source_file']==row['source_file'] and info['sha256']==sha256_file(archive):
            return archive,info
    url='https://www.kaggle.com/api/v1/datasets/download/kimdaegyeom/5g-traffic-datasets/'+urllib.parse.quote(row['source_file'],safe='')+'?datasetVersionNumber=1'
    tmp=archive.with_suffix('.partial')
    try:
        with urllib.request.urlopen(url,timeout=60) as r,tmp.open('wb') as f:
            n=0
            while True:
                b=r.read(min(1024*1024,cap-n+1))
                if not b:break
                n+=len(b)
                if n>cap:raise ValueError('Archive exceeds 128 MiB cap')
                f.write(b)
        with zipfile.ZipFile(tmp) as z:
            members=[m for m in z.infolist() if not m.is_dir()]
            if len(members)!=1 or not members[0].filename.endswith('.csv'):
                raise ValueError('Expected one CSV member')
            if members[0].file_size!=row['source_bytes']:
                raise ValueError('Inventory size and archive member size differ')
        tmp.replace(archive)
    finally:tmp.unlink(missing_ok=True)
    info={'source_file':row['source_file'],'requested_version':1,'sha256':sha256_file(archive),'archive_bytes':archive.stat().st_size}
    receipt.write_text(json.dumps(info,indent=2));return archive,info

def scan_capture(archive,row,root,receipt):
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    dest=root/(row['capture_id']+'.json');finger=root/(row['capture_id']+'.npy')
    if dest.exists() and finger.exists():
        cached=json.loads(dest.read_text())
        if cached.get('scan_version') in (1, SCAN_VERSION) and cached.get('archive_sha256')==receipt['sha256'] and cached.get('fingerprint_sha256')==sha256_file(finger):
            # Version 1 completed only with strict UTF-8, so its successful scans
            # necessarily had zero undecodable bytes. Preserve their fingerprints.
            cached.update(scan_version=SCAN_VERSION, undecodable_utf8_bytes=cached.get('undecodable_utf8_bytes',0), decoding_policy='UTF-8; undecodable bytes escaped as literal backslash-x hex; raw archive retained')
            dest.write_text(json.dumps(cached,indent=2))
            return dict(cached,split=row['split'])
    decode_stats={'bytes':0}
    def preserve_bad_bytes(error):
        decode_stats['bytes'] += error.end-error.start
        return codecs.backslashreplace_errors(error)
    codecs.register_error('qos_preserve_bytes',preserve_bad_bytes)
    required=['Time','Source','Destination','Protocol','Length','Info']
    total=bad=backwards=missing=0;previous=None;start=end=None;hashes=[];protocols={}
    with zipfile.ZipFile(archive) as z:
        with z.open(z.infolist()[0]) as stream:
            for df in pd.read_csv(stream,chunksize=50000,dtype=str,keep_default_na=False,encoding="utf-8",encoding_errors="qos_preserve_bytes"):
                if not set(required)<=set(df):raise ValueError('Required packet columns missing')
                total+=len(df)
                if total>20_000_000:raise ValueError('Capture row cap exceeded')
                t=pd.to_datetime(df['Time'],errors='coerce');bad+=int(t.isna().sum())
                good=t.dropna()
                if len(good):
                    backwards+=int((good.diff().dt.total_seconds()<0).sum())
                    if previous is not None and good.iloc[0]<previous:backwards+=1
                    previous=good.iloc[-1]
                    start=good.min() if start is None else min(start,good.min())
                    end=good.max() if end is None else max(end,good.max())
                missing+=int(df[required].eq('').any(axis=1).sum())
                for k,v in df['Protocol'].value_counts().items():protocols[k]=protocols.get(k,0)+int(v)
                # Excludes frame number: duplicated packets can be renumbered in exported files.
                hashes.append(pd.util.hash_pandas_object(df[required],index=False).to_numpy(dtype=np.uint64))
    if not hashes:raise ValueError('Empty capture')
    unique=np.unique(np.concatenate(hashes));del hashes
    np.save(finger,unique,allow_pickle=False)
    result={'undecodable_utf8_bytes':decode_stats['bytes'],
            'decoding_policy':'UTF-8; undecodable bytes escaped as literal backslash-x hex; raw archive retained',
            'scan_version':SCAN_VERSION,**row,'archive_sha256':receipt['sha256'],
            'fingerprint_sha256':sha256_file(finger),'rows':total,'timestamp_parse_failures':bad,
            'timestamp_backsteps':backwards,'rows_with_empty_required_fields':missing,
            'start_time':str(start),'end_time':str(end),'timezone':'unknown',
            'protocol_counts':protocols,'within_file_repeated_fingerprints':total-len(unique),
            'fingerprint_note':'64-bit fingerprints identify candidate duplicates, not collision-free proof.',
            'flow_id_status':'not constructed; dedicated port columns absent','training_ready':False}
    dest.write_text(json.dumps(result,indent=2));return result

def compare_captures(reports,root):
    pairs=[]
    for i,a in enumerate(reports):
        ah=np.load(Path(root)/(a['capture_id']+'.npy'),mmap_mode='r',allow_pickle=False)
        for b in reports[i+1:]:
            bh=np.load(Path(root)/(b['capture_id']+'.npy'),mmap_mode='r',allow_pickle=False)
            n=int(np.intersect1d(ah,bh,assume_unique=True).size)
            time_overlap=False
            if a['start_time']!='None' and b['start_time']!='None':
                time_overlap=max(pd.Timestamp(a['start_time']),pd.Timestamp(b['start_time']))<=min(pd.Timestamp(a['end_time']),pd.Timestamp(b['end_time']))
            if n or time_overlap:
                pairs.append({'capture_a':a['capture_id'],'capture_b':b['capture_id'],
                              'cross_split':a['split']!=b['split'],'shared_packet_fingerprints':n,
                              'overlapping_naive_time_ranges':bool(time_overlap),
                              'interpretation':'candidate for review; clock timezone and shared session not established'})
    return pairs


In [ ]:
reports,errors=[],[]
for i,row in enumerate(rows,1):
    print(f"[{i}/{len(rows)}] {row['source_file']}",flush=True)
    try:
        archive,receipt=acquire_capture(row,ARCHIVES)
        report=scan_capture(archive,row,SCANS,receipt)
        reports.append(report)
        print('Rows scanned:',report['rows'],'timestamp failures:',report['timestamp_parse_failures'],flush=True)
    except Exception as error:
        errors.append({'capture_id':row['capture_id'],'type':type(error).__name__,'message':str(error)[:300]})
        print('File failed; recorded error. Other files will continue.',flush=True)
    (ROOT/'progress.json').write_text(json.dumps({'completed':len(reports),'errors':errors},indent=2))


In [ ]:
pairs=compare_captures(reports,SCANS)
summary={'stage':'full_capture_acquisition_and_scan','manifest_sha256':actual,
         'expected_captures':len(rows),'completed_captures':len(reports),'errors':errors,
         'scanned_rows':sum(r['rows'] for r in reports),'reports':reports,'candidate_overlap_pairs':pairs,
         'license':'Unknown','training_ready':False,
         'next_step':'Review overlaps, packet attribution, and session provenance; decide grouping before flow extraction.',
         'limitations':['Clock timezone and device/session provenance not established.',
                        '64-bit matches are candidates requiring exact verification.',
                        'Absence of overlap does not prove session independence.',
                        'Application labels are capture-directory labels.']}
summary_path=ROOT/'capture_scan_summary.json'
summary_path.write_text(json.dumps(summary,indent=2))
import pandas as pd
pd.DataFrame(reports).drop(columns=['protocol_counts'],errors='ignore').to_csv(ROOT/'capture_scan_table.csv',index=False)
pd.DataFrame(pairs).to_csv(ROOT/'candidate_overlap_pairs.csv',index=False)
print('Completed:',len(reports),'of',len(rows),'Rows:',summary['scanned_rows'])
print('Candidate pairs:',len(pairs),'Errors:',len(errors))
print('Upload this file:',summary_path)


## Return one file
Upload **capture_scan_summary.json** from MyDrive → 5G_QoS_Research → data → full_capture_audit_v1. No need to paste raw rows or copy long output. If disconnected, reopen this notebook and run again: verified files and scans are reused. Interrupted individual downloads restart; completed files persist.
